### Kaggle-ready: Health Insurance Cross-Sell

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import joblib

# Reproducibility
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
# Use a numpy Generator (recommended over legacy global RNG)
rng = np.random.default_rng(seed)
random.seed(seed)
tf.random.set_seed(seed)

print('Libraries loaded — TensorFlow', tf.__version__)

In [ ]:
# Paths (Kaggle paths; fallback to local `data/` when not on Kaggle)
train_path = '/kaggle/input/health-insurance-cross-sell-prediction/train.csv'
test_path = '/kaggle/input/health-insurance-cross-sell-prediction/test.csv'
if not os.path.exists(train_path):
    train_path = 'data/train.csv'
    test_path = 'data/test.csv'

df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
# normalize column names to match Kaggle notebook conventions
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
test_df.columns = [c.strip().lower().replace(' ', '_') for c in test_df.columns]
print('Train shape:', df.shape, 'Test shape:', test_df.shape)
df.head()

In [ ]:
# Select features and target that match the competition
target_col = 'response'
features = [
    'gender',
    'age',
    'driving_license',
    'region_code',
    'previously_insured',
    'vehicle_age',
    'vehicle_damage',
    'annual_premium',
    'policy_sales_channel',
    'vintage'
]

# Build DataFrames for training and Kaggle test set
X = df[features].copy()
y = df[target_col].copy()
X_kaggle_test = test_df[features].copy()
print('Using features:', X.columns.tolist())

In [ ]:
# Preprocess train and test: map categorical, one-hot encode, and align columns
for col in ['gender', 'vehicle_damage']:
    if col in X.columns:
        X[col] = X[col].astype(str).str.lower()
        X_kaggle_test[col] = X_kaggle_test[col].astype(str).str.lower()

# gender -> {male:1, female:0}
if 'gender' in X.columns:
    X['gender'] = X['gender'].map({'male':1, 'female':0}).fillna(0).astype(int)
    X_kaggle_test['gender'] = X_kaggle_test['gender'].map({'male':1, 'female':0}).fillna(0).astype(int)

# vehicle_damage -> {yes:1, no:0}
if 'vehicle_damage' in X.columns:
    X['vehicle_damage'] = X['vehicle_damage'].map({'yes':1, 'no':0}).fillna(0).astype(int)
    X_kaggle_test['vehicle_damage'] = X_kaggle_test['vehicle_damage'].map({'yes':1, 'no':0}).fillna(0).astype(int)

# One-hot encode vehicle_age
categorical_cols = [c for c in ['vehicle_age'] if c in X.columns]
if categorical_cols:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    X_kaggle_test = pd.get_dummies(X_kaggle_test, columns=categorical_cols, drop_first=True)

# Align train and test columns (ensures same columns in same order)
X, X_kaggle_test = X.align(X_kaggle_test, join='left', axis=1, fill_value=0)
print('Shapes after encoding/alignment ->', X.shape, X_kaggle_test.shape)

# Keep DataFrame copies for later
X_df = X.copy()
X_kaggle_test_df = X_kaggle_test.copy()

# Train/validation split
X_train_df, X_val_df, y_train, y_val = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=seed
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df)
X_val = scaler.transform(X_val_df)
X_kaggle_test_scaled = scaler.transform(X_kaggle_test_df)
print('Train/Val shapes (scaled):', X_train.shape, X_val.shape)

In [ ]:
# Compute class weights to balance training
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print('Class weights:', class_weights)

# Build and train the model
tf.random.set_seed(seed)
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[es],
    class_weight=class_weights,
    verbose=1
)

# Evaluate on validation set
y_val_prob = model.predict(X_val).ravel()
y_val_pred = (y_val_prob >= 0.5).astype(int)
print('Accuracy:', accuracy_score(y_val, y_val_pred))
print('Precision:', precision_score(y_val, y_val_pred, zero_division=0))
print('Recall:', recall_score(y_val, y_val_pred, zero_division=0))
print('F1-score:', f1_score(y_val, y_val_pred, zero_division=0))
print(classification_report(y_val, y_val_pred))

# Plot training history: loss and accuracy
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history.get('loss', []), label='train_loss')
plt.plot(history.history.get('val_loss', []), label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(history.history.get('accuracy', []), label='train_acc')
plt.plot(history.history.get('val_accuracy', []), label='val_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Save model and scaler for reproducibility
model.save('kaggle_model.h5')
joblib.dump(scaler, 'kaggle_scaler.joblib')
print('Saved kaggle_model.h5 and kaggle_scaler.joblib')

# Predict probabilities for Kaggle submission
test_prob = model.predict(X_kaggle_test_scaled).ravel()

# Optional: verify sample submission format (if available)
sample_sub_path = '/kaggle/input/health-insurance-cross-sell-prediction/sample_submission.csv'
if os.path.exists(sample_sub_path):
    sample_sub = pd.read_csv(sample_sub_path)
    print('Sample submission format:')
    print(sample_sub.head())
    print('Expected columns:', sample_sub.columns.tolist())

# Create submission file using probabilities (preferred for ROC-AUC)
submission = pd.DataFrame({'id': test_df['id'], 'Response': test_prob})
submission.to_csv('submission.csv', index=False)
print('submission.csv created successfully!')
print(submission.head())

# Optional: simple permutation importance on validation set
baseline_f1 = f1_score(y_val, y_val_pred)
feature_names = X_df.columns.tolist()
importances = []
for col in feature_names:
    X_val_perm = X_val_df.copy()
    X_val_perm[col] = rng.permutation(X_val_perm[col].values)
    X_val_perm_scaled = scaler.transform(X_val_perm)
    preds = (model.predict(X_val_perm_scaled).ravel() >= 0.5).astype(int)
    f1p = f1_score(y_val, preds)
    importances.append(baseline_f1 - f1p)
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)

print('\nTop 20 important features:')
print(imp_df.head(20))

plt.figure(figsize=(10,6))
topn = imp_df.head(15)[::-1]
plt.barh(topn['feature'], topn['importance'])
plt.xlabel('Permutation Importance (drop in F1)')
plt.ylabel('Feature')
plt.title('Top 15 Important Features')
plt.grid(True)
plt.show()